# PYNQ-Z2 — Electrode Resistance Measurement

## Circuit
```
3.3V ──[R_known = 10kΩ]──┬──[ELECTRODE]── GND
                          │
                         A0  (Arduino header, XADC)
```

## How it works
The circuit is a **voltage divider**. The ADC reads the voltage at A0, which depends on both resistors:
```
V_A0 = 3.3 × R_electrode / (R_known + R_electrode)
```
Rearranged to solve for the electrode resistance:
```
R_electrode = R_known × V_adc / (V_FS − V_adc)
```
where **V_FS** is the XADC full-scale voltage, calibrated once using two equal known resistors.

## Workflow
1. **Cell 1** — Load overlay & drivers
2. **Cell 2** — Calibrate V_FS *(use two equal 10kΩ resistors, no electrode)*
3. **Cell 3** — Define helpers
4. **Cell 4** — Set baseline *(electrode connected, at rest)*
5. **Cell 5** — Live monitoring
6. **Cell 6** — Plot results

In [ ]:
# ── Cell 1: Load overlay & drivers ───────────────────────────────────────────
from pynq import Overlay
from pynq.lib.arduino import Arduino_Analog
from collections import deque
import matplotlib.pyplot as plt
import time

ol = Overlay('base.bit')
analog = Arduino_Analog(ol.iop_arduino.mb_info, [0])

print('Overlay loaded  ✓')
print('Analog driver ready  ✓')

In [ ]:
# ── Cell 2: Calibrate V_FS ───────────────────────────────────────────────────
#
# V_FS is the XADC effective full-scale voltage. It is a hardware property
# of the board and does NOT change with the electrode.
#
# HOW TO CALIBRATE:
#   1. Remove the electrode
#   2. Connect two equal 10kΩ resistors (voltage divider: 3.3V→10kΩ→A0→10kΩ→GND)
#   3. Run this cell — it measures the midpoint and computes V_FS = 2 × V_mid
#   4. Hardcode the printed value below
#
# If you have already calibrated, SKIP this cell and set V_FS manually.

R_KNOWN = 10_000   # Ω — your fixed top resistor

print('CALIBRATION: Ensure two equal 10kΩ resistors are connected (no electrode).')
input('Press Enter when ready...')

cal_samples = [float(analog.read()) for _ in range(20)]
v_cal = sum(cal_samples) / len(cal_samples)

# With two equal resistors, V_adc = V_FS / 2  →  V_FS = 2 × V_adc
V_FS = 2 * v_cal

print(f'\nCalibration V_adc : {v_cal:.4f} V')
print(f'Effective V_FS    : {V_FS:.4f} V')
print(f'\n>>> Hardcode this: V_FS = {V_FS:.4f}')
print('Now remove the calibration resistors and reconnect the electrode.')

In [ ]:
# ── Cell 3: Configuration & Helper functions ──────────────────────────────────

R_KNOWN   = 10_000   # Ω   — fixed 10kΩ top resistor
V_FS      = 1.3391   # V   — ← paste your calibrated value here
N_AVG     = 10       # number of ADC samples averaged per reading
WINDOW    = 5        # moving average window size (smoothing)
THRESHOLD = 2.0      # % change below which we treat reading as noise


def calc_resistance(v_adc):
    """
    Convert ADC voltage → electrode resistance (Ω).

    Formula (voltage divider rearranged):
        R_electrode = R_known × V_adc / (V_FS − V_adc)
    """
    v_adc = max(0.001, min(v_adc, V_FS * 0.999))
    return R_KNOWN * v_adc / (V_FS - v_adc)


def read_resistance(n=N_AVG):
    """Average n ADC readings to reduce noise, return resistance in Ω."""
    samples = [calc_resistance(float(analog.read())) for _ in range(n)]
    return sum(samples) / len(samples)


print('Configuration:')
print(f'  R_KNOWN   = {R_KNOWN} Ω')
print(f'  V_FS      = {V_FS} V')
print(f'  N_AVG     = {N_AVG} samples')
print(f'  WINDOW    = {WINDOW} readings')
print(f'  THRESHOLD = {THRESHOLD} %')
print('\nHelpers ready  ✓')

In [ ]:
# ── Cell 4: Set Baseline ──────────────────────────────────────────────────────
#
# Reconnect electrode. Keep it in its RESTING state (no contact/pressure).
# This reading becomes the reference point for ΔR calculations.

print('Setting baseline — keep electrode in RESTING state (no contact)...')
time.sleep(2)

R_BASELINE = read_resistance(n=30)   # 30-sample average for stable baseline

print(f'\nBaseline resistance : {R_BASELINE:.1f} Ω')
print(f'                    : {R_BASELINE/1000:.3f} kΩ')
print('\nBaseline set  ✓  Now you can touch / press / wet the electrode.')

In [ ]:
# ── Cell 5: Live Monitoring ───────────────────────────────────────────────────
#
# Reads electrode resistance continuously.
# Two-layer noise reduction:
#   1. Average N_AVG ADC samples per reading   → removes ADC noise
#   2. Moving average over WINDOW readings      → removes reading-to-reading jitter

NUM_SAMPLES  = 100    # total number of readings to collect
DELAY        = 0.2    # seconds between readings

window       = deque(maxlen=WINDOW)
log_r        = []     # smoothed resistance log (for plotting)
log_dr       = []     # ΔR log

print(f"{'#':>4}  {'R (kΩ)':>9}  {'ΔR (Ω)':>10}  {'ΔR%':>8}  State")
print('─' * 57)

for i in range(NUM_SAMPLES):
    r_raw = read_resistance()       # averaged single reading
    window.append(r_raw)
    r_smooth = sum(window) / len(window)   # smoothed

    dR  = r_smooth - R_BASELINE
    dPc = dR / R_BASELINE * 100

    if   abs(dPc) < THRESHOLD:  state = '── RESTING'
    elif dPc < 0:               state = f'↓ CONTACT   (−{abs(dPc):.1f}%)'
    else:                       state = f'↑ INCREASED (+{dPc:.1f}%)'

    log_r.append(r_smooth)
    log_dr.append(dR)

    print(f"{i+1:>4}  {r_smooth/1000:>9.3f}  {dR:>+10.1f}  {dPc:>+7.1f}%  {state}")
    time.sleep(DELAY)

print('\nDone. Run Cell 6 to plot results.')

In [ ]:
# ── Cell 6: Plot Results ──────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('PYNQ-Z2 — Electrode Resistance Measurement', fontsize=14, fontweight='bold')

samples = range(1, len(log_r) + 1)

# ── Top plot: Resistance ──
ax1.plot(samples, [r/1000 for r in log_r], color='royalblue', linewidth=1.5, label='R electrode')
ax1.axhline(R_BASELINE/1000, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({R_BASELINE/1000:.3f} kΩ)')
ax1.set_ylabel('Resistance (kΩ)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_title('Electrode Resistance over Time')

# ── Bottom plot: ΔR ──
colors = ['tomato' if d < 0 else 'mediumseagreen' for d in log_dr]
ax2.bar(samples, log_dr, color=colors, alpha=0.75, width=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('ΔR (Ω)')
ax2.set_xlabel('Sample index')
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_title('Change in Resistance from Baseline (red = contact/decrease, green = increase)')

plt.tight_layout()
plt.savefig('electrode_resistance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as electrode_resistance.png')